# Target Projection Mechanics & Module Selection Strategies

In a modern Transformer architecture (like Llama 3 or Mistral), linear layers appear in two distinct blocks per transformer layer:

* **Multi-Head Self-Attention (MHSA) / Grouped-Query Attention (GQA) block**
* **Feed-Forward Network (FFN / MLP) block**

Choosing which linear projections to attach LoRA adapters to directly impacts parameter efficiency, adaptation quality, and VRAM overhead.

## Transformer Linear Module Taxonomy

Inside every Transformer layer, there are 7 primary linear projection matrices:

```
                  ┌──────────────────────────────────────────────┐
                  │            Transformer Layer                 │
                  └──────────────────────┬───────────────────────┘
                                         │
              ┌──────────────────────────┴──────────────────────────────┐
              │                                                         │
              ▼                                                         ▼
   [ Attention Block (GQA) ]                                    [ MLP / FFN Block ]
   ┌──────────────────────────────┐                     ┌──────────────────────────────┐
   │ Query Proj  : q_proj (d x d) │                     │ Gate Proj : gate_proj (d x m)│
   │ Key Proj    : k_proj (d x k) │                     │ Up Proj   : up_proj   (d x m)│
   │ Value Proj  : v_proj (d x k) │                     │ Down Proj : down_proj (m x d)│
   │ Output Proj : o_proj (d x d) │                     └──────────────────────────────┘
   └──────────────────────────────┘
```

Where:
* **$d$** is the hidden dimension (e.g., $4096$)
* **$m$** is the intermediate MLP dimension (e.g., $14336$ for SwiGLU architectures)

## How to decide LoRA target modules and rank

In a standard Transformer block, each layer contains:

- Attention projections: $W_q, W_k, W_v$ for token interaction and $W_o$ for attention aggregation
- MLP / feed-forward projections: $W_{\text{gate}}, W_{\text{up}}$ for information retrieval and feature expansion, and $W_{\text{down}}$ for compression back to hidden size

## Where does the adaptation live?

| Focus area | Typical target modules | Best for |
|---|---|---|
| Attention only | $W_q, W_v$ | Routing, context, roleplay, instruction following |
| MLP only | $W_{\text{gate}}, W_{\text{up}}, W_{\text{down}}$ | Domain facts, terminology, code generation, structured syntax |

This split reflects where different kinds of knowledge and behavior are often stored in a Transformer.

## Evolution of Targeting: Attention-Only vs. All-Linear

### Legacy Approach (LoRA Original Paper, 2021)

**Target Modules:** $q\_proj$ and $v\_proj$ only

**Rationale:** Minimizing parameter count to the absolute limit

**System Failure Mode:** Constraining LoRA to $q$ and $v$ severely limits model capacity when adapting to complex downstream domain tasks:
* Teaching medical terminology
* Novel code syntaxes
* Complex instruction formats

The MLP layers, which store a vast amount of parametric knowledge, remain completely rigid.

---

### Modern SOTA Approach (All-Linear Targeting)

**Target Modules:** `["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]`

**System Insight:** Research (e.g., QLoRA paper) demonstrates that targeting all linear layers with a lower rank (e.g., $r=8$ or $r=16$) yields significantly higher benchmark performance than targeting only Attention projections with a higher rank (e.g., $r=64$).

## Module Selection Tradeoff Matrix

| Targeted Modules | Parameter Count (Relative) | VRAM Overhead | Convergence Speed | Fine-Tuning Capability / Domain Adaptation |
|---|---|---|---|---|
| `q_proj`, `v_proj` | Very Low ($\sim 0.05\%$) | Minimal | Slow | Low (Good only for minor prompt style alignment) |
| All Attention (q, k, v, o) | Low ($\sim 0.1\%$) | Low | Moderate | Moderate (Good for task alignment and instruction following) |
| MLP Only (gate, up, down) | High ($\sim 0.4\%$) | Moderate | Fast | High (Strong for factual knowledge adaptation) |
| All-Linear (Attention + MLP) | Optimal ($\sim 0.5\%$) | Low-Moderate | Fastest | **Optimal** |

## Rank Selection & Allocation Strategy

When designing your LoRA configuration, follow this principle:

### Prefer All-Linear with Low Rank over Selective-Linear with High Rank

**Bad Config:** Target `q_proj`, `v_proj` with $r = 64$ ($\approx 10.4\text{M}$ params)

**Good Config:** Target All 7 Linear Modules with $r = 8$ or $r = 16$ ($\approx 20\text{M}$ params)

---

### When to Increase Rank ($r$)

* **$r = 8 \text{ to } 16$:** Ideal for structured JSON output, classification, tool calling, and instruction tuning
* **$r = 32 \text{ to } 64$:** Required for complex code generation, new natural language translation, or heavy domain adaptation (e.g., financial/legal parsing)
* **$r > 64$:** Almost always yields diminishing returns; causes overfitting and wastes VRAM. Consider full fine-tuning or Continued Pre-Training instead

# Merging the parameters

## Adapter Lifecycle, Saving, Loading, and Precision-Safe Merging

Operating LoRA adapters in production requires strict adherence to serialization protocols, parameter isolation, and numerical precision rules during weight merging.

---

### Adapter Serialization vs. Full Model Checkpoints

When fine-tuning completes, saving the entire base model plus adapters is inefficient and redundant.

**Saved Adapter Artifact Architecture**

A standard PEFT adapter directory requires only two tiny files ($\sim 10\text{ MB}$ to $100\text{ MB}$ total depending on $r$):

* **`adapter_config.json`:** Metadata defining adapter topology
  - Target modules list (`q_proj`, `v_proj`, `gate_proj`, etc.)
  - Rank ($r$) and Scaling Factor ($\alpha$)
  - Base model architecture type & dropout rates

* **`adapter_model.safetensors`:** Tensors containing only matrices $A$ and $B$ for all targeted layers

```
/my-lora-adapter/
├── adapter_config.json         (~1 KB)
├── adapter_model.safetensors   (~20-80 MB)
└── README.md
```

**Safety Standard: safetensors over .bin / PyTorch Pickles**

* **Zero-Copy Memory Mapping (mmap):** safetensors maps data directly from disk into GPU memory without intermediate host RAM allocations
* **Arbitrary Code Execution Prevention:** PyTorch `.bin` files use Python's pickle, which can execute arbitrary code upon deserialization. safetensors is purely a strict buffer byte-parser

---

### The Mechanics of Precision-Safe Weight Merging

Merging LoRA weights ($\Delta W = \frac{\alpha}{r} B \cdot A$) into base weights ($W_0$) seems algebraically simple:

$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} (B \cdot A)$$

However, performing this operation naively across mixed-precision representations introduces Numerical Underflow/Overflow and Dequantization Drift.

---

### Numerical Hazards During Merging

**Failure Mode 1: Accumulation Truncation in FP16 / BF16**

If base weights $W_0$ and adapter matrices $A, B$ are stored in 16-bit floating-point formats (FP16 or BF16):

* FP16 has only 10 bits of mantissa (significand) precision
* When calculating $\Delta W = B \cdot A$, small parameter values multiplied together can fall below the FP16 minimum normal threshold ($\sim 6.10 \times 10^{-5}$), causing them to abruptly underflow to 0.0
* Adding small FP16 deltas to larger FP16 base weights causes loss of precision due to cataclysmic mantissa alignment truncation

**Mitigation Protocol: FP32 Upcasting**

Always perform weight merging in 32-bit Single Precision (FP32), regardless of the precision of the saved weights:

```
              ┌─────────────────────────────────────┐
              │   Base Weights W_0 (16-bit BF16)    │
              └──────────────────┬──────────────────┘
                                 │
                             [ Upcast ]
                                 │
                                 ▼
              ┌─────────────────────────────────────┐
              │    Base Weights W_0 (32-bit FP32)   │
              └──────────────────┬──────────────────┘
                                 │
                                 │  ◄── Add [ (alpha / r) * (B_fp32 * A_fp32) ]
                                 ▼
              ┌─────────────────────────────────────┐
              │    Merged Weights W_m (32-bit FP32) │
              └──────────────────┬──────────────────┘
                                 │
                             [ Downcast ]
                                 │
                                 ▼
              ┌─────────────────────────────────────┐
              │  Export Artifact (16-bit BF16/FP16) │
              └─────────────────────────────────────┘
```

**Process Steps:**

1. Load base model $W_0$ and cast to `torch.float32`
2. Load adapter matrices $A$ and $B$ and cast to `torch.float32`
3. Compute $\Delta W_{\text{fp32}} = \left(\frac{\alpha}{r}\right) (B_{\text{fp32}} \cdot A_{\text{fp32}})$
4. Compute $W_{\text{merged\_fp32}} = W_{0\text{\_fp32}} + \Delta W_{\text{fp32}}$
5. Downcast $W_{\text{merged\_fp32}}$ back to `torch.bfloat16` or `torch.float16` for final export

---

**Failure Mode 2: Merging Directly into Quantized Base Weights**

Attempting to directly add floating-point adapter deltas $\Delta W$ to a 4-bit quantized base model ($W_{\text{NF4}}$ or $W_{\text{INT4}}$) without un-quantizing first is a fatal error:

* Quantized weights are discrete integers scaled by block-wise floating-point scale factors
* Adding un-quantized continuous floating-point values directly breaks the quantization grid and destroys the parameter distribution

**Rule:** Dequantize the 4-bit base model to FP32 first, merge $\Delta W$ in FP32, then re-quantize the final merged model if low-bit deployment is required